# Live Demo: Gender Wage Gap — ML Analysis

The Random Forest model was trained using **PySpark MLlib** in `analysis/ml_rf_antara.py`.

To keep the live demo fast, this notebook loads the saved PySpark outputs instead of retraining the model.


In [2]:
import pandas as pd
import plotly.express as px
from pathlib import Path
import sys

root = Path.cwd()
while not (root / "_quarto.yml").exists() and root != root.parent:
    root = root.parent

sys.path.insert(0, str(root / "analysis"))

from utils import PROCESSED, apply_theme

print(f"Repo root: {root}")
print(f"Processed folder: {PROCESSED}")

Repo root: /home/ubuntu/repoad688-employability-sum26-groupB
Processed folder: /home/ubuntu/repoad688-employability-sum26-groupB/data/processed


## Load PySpark Random Forest outputs

In [7]:
rf_summary = pd.read_csv(PROCESSED / "rf_model_summary_antara.csv")
rf_imp = pd.read_csv(PROCESSED / "rf_feature_importance_antara.csv")
pdp = pd.read_csv(PROCESSED / "rf_partial_dependence_age_gender_antara.csv")

display(rf_summary)
rf_imp.head(10)

,metric,value
0,Train R²,0.270347
1,Test R²,0.235658
2,Sample Size,200517.000000
3,Num Trees,100.000000
4,Max Depth,10.000000
5,Default Parallelism,2.000000
6,Shuffle Partitions,8.000000


,feature,importance,is_gender
0,AGE,0.245338,False
1,OCC_GROUP_3000,0.137236,False
2,OCC_GROUP_0,0.119030,False
3,SEX_LABEL_Male,0.056332,True
4,OCC_GROUP_2100,0.056289,False
5,SEX_LABEL_Female,0.045806,True
6,OCC_GROUP_1000,0.043695,False
7,OCC_GROUP_400,0.041064,False
8,OCC_GROUP_800,0.025374,False
9,OCC_GROUP_100,0.022500,False


## Feature importance

In [ ]:
top20 = rf_imp.head(20).copy()

def clean_feature_label(f):
    if f == "AGE":
        return "Age"
    elif 'SEX_LABEL' in f:
        return "Gender"
    elif f.startswith("STATE_NAME_"):
        return f"State: {f.replace('STATE_NAME_', '')}"
    elif f.startswith("RACE_LABEL_"):
        return f"Race: {f.replace('RACE_LABEL_', '')}"
    elif f.startswith("OCC_GROUP_"):
        return f"Occupation group {f.replace('OCC_GROUP_', '')}"
    elif f.startswith("IND_GROUP_"):
        return f"Industry group {f.replace('IND_GROUP_', '')}"
    return f

top20["label"] = top20["feature"].apply(clean_feature_label)
top20["is_gender"] = top20["label"] == "Gender"

top15 = (
    top20
    .groupby("label", as_index=False)
    .agg(importance=("importance", "sum"), is_gender=("is_gender", "max"))
    .sort_values("importance", ascending=False)
    .head(15)
)

fig_lollipop = go.Figure()
fig_lollipop.add_trace(go.Scatter(
    x=top15['importance'], y=top15['label'],
    mode='markers',
    marker=dict(
        size=14,
        color=top15['is_gender'].map({True: '#d7191c', False: '#2c7bb6'}),
    ),
))
for _, row in top15.iterrows():
    fig_lollipop.add_shape(
        type='line', x0=0, x1=row['importance'],
        y0=row['label'], y1=row['label'],
        line=dict(color='#d7191c' if row['is_gender'] else '#cccccc', width=2),
    )
fig_lollipop.update_layout(
    title='PySpark Random Forest Feature Importance (Top 15 of 172 features)',
    xaxis_title='Importance',
    yaxis_title='',
    yaxis=dict(categoryorder='total ascending'),
)
apply_theme(fig_lollipop)
fig_lollipop.show()

## Gender rank

In [5]:
gender_rows = rf_imp[rf_imp["is_gender"] == True].copy()

if len(gender_rows) > 0:
    first_gender_rank = gender_rows.index[0] + 1
    total_features = len(rf_imp)
    print(f"Gender first appears at rank #{first_gender_rank} out of {total_features} features.")
    display(gender_rows.head())
else:
    print("No gender feature found.")

Gender first appears at rank #4 out of 20 features.


,feature,importance,is_gender
3,SEX_LABEL_Male,0.056332,True
5,SEX_LABEL_Female,0.045806,True


## Predicted wage by age and gender

In [6]:
fig = px.line(
    pdp,
    x="AGE",
    y="predicted_wage",
    color="SEX",
    title="PySpark Random Forest: Predicted Wage by Age and Gender",
    labels={
        "AGE": "Age",
        "predicted_wage": "Predicted Annual Wage ($)",
        "SEX": "Gender"
    }
)

apply_theme(fig)
fig.show()

## Key takeaway

The PySpark Random Forest confirms that gender remains a strong predictor of wage after accounting for age, race, state, occupation, and industry.
